In [0]:
%pip install pypdf
%restart_python

In [0]:
%restart_python

In [0]:
from pyspark.sql import functions as f
import pypdf
import json
import re
import os

In [0]:
arquivos = {}
path = "/Workspace/Users/alexandre.cesardf@hotmail.com/pipeline_peticoes_previdenciarias/Files"
files =  os.listdir(path)
for file in files:
    if "rg" in file.lower():
        arquivos["rg"] = file
    if "residencia" in file.lower():
        arquivos["residencia"] = file
    if "laudo" in file.lower():
        arquivos["laudo"] = file
    if "indeferimento" in file.lower():
        arquivos["indeferimento"] = file

In [0]:
def extrair_campos(texto, regexes):
    resultado = {}

    for campo, regex in regexes.items():
        match = re.search(regex, texto, re.IGNORECASE | re.DOTALL)

        resultado[campo] = (
            match.group(1).strip()
            if match
            else None
        )

    return resultado

In [0]:
for key, item in arquivos.items():
    print(key, item)

    with open("/Workspace/Users/alexandre.cesardf@hotmail.com/pipeline_peticoes_previdenciarias/configs/documentos.json", encoding="utf-8") as file:
        regexes = json.load(file)[key]
    
    reader = pypdf.PdfReader(path+item)
    pdf = reader.pages[0].extract_text()
        
    try:
        resultado = extrair_campos(pdf, regexes)
        df = spark.createDataFrame([resultado])
        if key != "laudo":
            df = df.withColumn("cpf", f.lit(f.col("cpf")))
        df.write.mode("overwrite").saveAsTable(f"workspace.bronze.{key}")
    except:
        print(f"Não foi possivel criar a tabela {key}, arquivo -> {item}")